# Example Literature Project

## Helper Functions

In [ ]:
import json
import textwrap
from pathlib import Path
from time import perf_counter

from kilin.conversations import ConversationStore
from kilin.rag import RAGSession


def _print_wrapped(label, value, width=79):
    """Print text with a fixed display width, preserving paragraphs."""
    print(f"{label}:")
    for line in str(value).splitlines() or [""]:
        if not line.strip():
            print()
        else:
            print(textwrap.fill(
                line, width=width, break_long_words=False,
                break_on_hyphens=False,
            ))
    print("")


def create_conversation(project_dir):
    """Create an independent conversation and its active RAG session.

    Parameters
    ----------
    project_dir : str or Path
        Existing KíLín project directory.

    Returns
    -------
    tuple
        Conversation store, conversation record, and active session.
        Close the session after the last question.

    """
    store = ConversationStore(Path(project_dir))
    conversation = store.create()
    path = store.state_path(conversation)
    session = RAGSession(Path(project_dir), conversation_path=path)
    print(f"Conversation: {path}")
    return store, conversation, session


def ask_question(session, question):
    """Ask the active model and print a width-limited exchange.

    Parameters
    ----------
    session : RAGSession
        Active conversation session.
    question : str
        User question.

    Returns
    -------
    str
        Assistant response.

    """
    print("\n" + "=" * 79)
    _print_wrapped("USER", question)
    start = perf_counter()
    try:
        answer = session.ask(question)
    except Exception:
        print(f"Elapsed before failure: {perf_counter() - start:.3f} s")
        raise
    _print_wrapped("ASSISTANT", answer)
    print(f"Wall time: {perf_counter() - start:.3f} s")
    return answer


def _print_retrieved_chunks(stages, width=79):
    """Display retrieved passages recorded in the native trace.

    Parameters
    ----------
    stages : dict
        Stages of one recorded conversation turn.
    width : int, optional
        Maximum display width.

    """
    retrieval = stages.get("03_retrieval", {})
    operations = retrieval.get("operations", {})

    print("\nRETRIEVED CHUNKS")

    found = 0

    def visit(value, path):
        nonlocal found

        if isinstance(value, dict):
            text = next(
                (
                    value[key]
                    for key in (
                        "text", "content", "chunk_text",
                        "passage", "snippet",
                    )
                    if isinstance(value.get(key), str)
                    and value[key].strip()
                ),
                None,
            )

            if text is not None:
                found += 1
                print(f"\n  CHUNK {found}")
                print(f"  Trace location: {path}")

                for key, item in value.items():
                    if key in (
                        "text", "content", "chunk_text",
                        "passage", "snippet",
                    ):
                        continue

                    if isinstance(item, (str, int, float, bool)):
                        _print_wrapped(f"  {key}", item, width)

                _print_wrapped("  PASSAGE", text, width)
                return

            for key, child in value.items():
                visit(child, f"{path}/{key}")

        elif isinstance(value, list):
            for index, child in enumerate(value):
                visit(child, f"{path}/{index}")

    for name, operation in operations.items():
        if "retrieval" not in name.lower():
            continue

        visit(operation.get("details", {}), name)

    if not found:
        print("  No passage text found in the retrieval trace.")


def inspect_conversation(
    conversation_path,
    turn_number=None,
    show_chunks=True,
):
    """Display routing, retrieval, model calls, tools, and trace timings.

    Parameters
    ----------
    conversation_path : str or Path
        Native KíLín trace JSON file.
    turn_number : int, optional
        One-based turn index, or None for all turns.
    show_chunks : bool, optional
        Whether to display passages recorded in the retrieval trace.
        Defaults to True.

    Returns
    -------
    dict
        Unmodified native trace record.

    """
    with Path(conversation_path).open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = list(trace["sessions"]["05_conversations"]["turns"].items())
    if turn_number is not None:
        if not 1 <= turn_number <= len(turns):
            raise ValueError("Invalid turn number.")
        turns = [turns[turn_number - 1]]

    def operation(stages, stage_name, operation_name):
        return (stages.get(stage_name, {})
                .get("operations", {}).get(operation_name, {}))

    def show_fields(heading, data, keys):
        print(f"\n{heading}")
        for key in keys:
            if key in data:
                _print_wrapped(f"  {key}", data[key])

    for turn_id, turn in turns:
        stages = turn.get("stages", {})
        print("\n" + "=" * 79)
        print(f"TURN: {turn_id}")
        print("=" * 79)

        question = operation(stages, "01_question", "01_turn")
        response = operation(
            stages, "01_question", "09_conversation_turn_representation"
        )
        show_fields("EXCHANGE", question.get("details", {}), ["question"])
        show_fields("ANSWER", response.get("details", {}), ["summary"])

        decision = operation(
            stages, "01_question", "05_intent_classification_decided"
        ).get("details", {})
        show_fields("INTENT", decision.get("classification", {}), [
            "intent", "standalone", "refers_back", "related_turn_ids"
        ])
        show_fields("ROUTING", decision, ["shortcut", "fallback"])
        show_fields("RESOLVED POLICY", decision.get("resolved", {}), [
            "history_policy", "retrieval_mode", "retrieval_strategy"
        ])
        settings = operation(
            stages, "01_question", "02_turn_settings_snapshot"
        ).get("details", {})
        show_fields("EFFECTIVE SETTINGS", settings, [
            "project_type_id", "backend", "chat_model", "router_model",
            "thinking", "temperature", "max_new_tokens", "retrieval_count"
        ])
        retrieval = operation(
            stages, "03_retrieval", "02_retrieval_result"
        )
        show_fields("RETRIEVAL", retrieval.get("details", {}), [
            "mode", "strategy", "evidence_count", "duration_ms"
        ])
        if show_chunks:
            _print_retrieved_chunks(stages)

        calls = []
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                if name == "01_llm_call_input" or name == "02_llm_call_output":
                    calls.append((stage_name, name, entry))
                elif "tool" in name.lower():
                    print(f"\nTOOL OPERATION: {stage_name}/{name}")
                    show_fields("  DETAILS", entry.get("details", {}), [
                        "tool", "tool_name", "status", "duration_ms"
                    ])
        for stage_name, name, entry in calls:
            details = entry.get("details", {})
            show_fields(f"MODEL CALL: {stage_name}/{name}", details, [
                "call_id", "model_id", "backend", "device", "thinking",
                "input_token_count", "prompt_token_count",
                "output_token_count", "finish_reason", "reply_duration_ms"
            ])
            show_fields("GENERATION SETTINGS", details.get("generation", {}), [
                "temperature", "max_new_tokens", "thinking"
            ])

        print("\nOPERATION TIMINGS (ms)")
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                timing = entry.get("timing", {})
                duration = timing.get("operation_duration_ms")
                if duration is not None:
                    print(f"  {stage_name}/{name}: {duration:.3f}")

    return trace


/Users/giovannivolpe/Documents/GitHub/KiLin/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Create Empty Literature Project

In [4]:
from pathlib import Path

from kilin.workspace import prepare_sources_workspace

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run this notebook from within the KíLín repository.")
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = prepare_sources_workspace(
    [],
    "literature",
    workspace_directory=project_dir,
)

print(f"Created literature project: {project.root}")
print(f"Source directory: {project.source}")

Created literature project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature
Source directory: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/sources


## First Conversation without Sources

In [5]:
store, conversation, session = create_conversation(project_dir)

Fetching 13 files: 100%|██████████| 13/13 [00:00<00:00, 3491.67it/s]
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 2391.67it/s]


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-10-30T10-03-41.json


In [6]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
hello

ASSISTANT:
Hello!

Wall time: 1.527 s

TURN: turn_0001

EXCHANGE
  question:
hello


ANSWER
  summary:
Hello!


INTENT
  intent:
small_talk

  standalone:
True

  refers_back:
False

  related_turn_ids:
[]


ROUTING
  shortcut:
(hi|hello|hey|hiya|hej|yo|greetings|good (morning|afternoon|evening|day))(
there| kilin)?

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
5


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.007


MODEL CALL: 04_llm_input/01_llm_call_input
  call_id:
llm-call_9badae936e85

  model_id:
Qwen/Qwen3-8B-MLX-4bit

  backend:
mlx

  device:
metal

  thinking:
False

  input_token_count:
116

  prompt_token_count:
116


GENERATION SETTINGS
  temperature:
0.0

  max_new_tokens:

In [7]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What can you help me with?

ASSISTANT:
<think>
Okay, the user asked, "What can you help me with?" and I need to respond based
on the project evidence provided. Let me check the evidence again. The user
mentioned that I should only use the supplied project evidence and cite sources
like [1] and [2]. They also said not to include a reference list and to say
when evidence is insufficient.

First, I need to recall what the project evidence includes. From the previous
conversation, the user provided some evidence, but I don't have the actual
content here. Wait, in the initial interaction, the user said "hello" and I
responded, then they asked about what I can help with. The user's latest
message is the same as before, but maybe the project evidence is the initial
messages. Wait, maybe the user is referring to a specific set of evidence they
provided earlier. But in the current context, the user hasn't provided any
project evidence. Wait, the user's first message was "hello", then "Wh

In [8]:
answer = ask_question(session, 'What is Brownian motion?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
What is Brownian motion?

ASSISTANT:
<think>
Okay, the user is asking about Brownian motion. Let me recall what I know.
Brownian motion is the random movement of particles in a fluid, right? It's
named after Robert Brown, who observed it in 1827. The key points are that it's
caused by collisions with molecules, which are in constant motion. This
phenomenon is important in various fields like physics, chemistry, and finance.

Wait, the user wants the answer based on the supplied project evidence. But
looking back, the previous messages don't have any project evidence. The user
just asked about Brownian motion, and the assistant answered based on general
knowledge. However, the user's initial instruction said to answer only from the
supplied project evidence. Since there's no evidence provided, the assistant
should mention that.

In the previous interaction, the user asked "What can you help me with?" and
the assistant mentioned needing project evidence. Then the user asked about


In [9]:
answer = ask_question(session, 'Which articles are available in this project?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Which articles are available in this project?

ASSISTANT:
This project has **0 source files**, all with searchable text.

**By type**


**0 files**


Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Wall time: 8.943 s

TURN: turn_0004

EXCHANGE
  question:
Which articles are available in this project?


ANSWER
  summary:
This project has **0 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
5


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.003


TOOL OPERATION: 01_questio

In [10]:
answer = ask_question(session, 'Summarize the articles in this project.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Summarize the articles in this project.

ASSISTANT:
This project has **0 source files**, all with searchable text.

**By type**


Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Wall time: 9.096 s

TURN: turn_0005

EXCHANGE
  question:
Summarize the articles in this project.


ANSWER
  summary:
This project has **0 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
5


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.003


TOOL OPERATION: 01_question/08_tool_project_inventor

In [11]:
answer = ask_question(session, 'Find articles about neural networks in this project.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Find articles about neural networks in this project.

ASSISTANT:
Found **0 sentences** about “neural networks” in **0 of 0 files**.
No sentence in scope is about this topic.

Wall time: 12.417 s

TURN: turn_0006

EXCHANGE
  question:
Find articles about neural networks in this project.


ANSWER

INTENT
  intent:
topic_listing

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen/Qwen3-8B-MLX-4bit

  thinking:
True

  temperature:
0.0

  max_new_tokens:
8192

  retrieval_count:
5


RETRIEVAL
  mode:
none

  strategy:
hybrid

  evidence_count:
0

  duration_ms:
0.002


TOOL OPERATION: 01_question/08_tool_plan_proposed

  DETAILS
  tool:
find_passages


TOOL OPERATION: 01_question/09_tool_plan_decided

  DETAILS

OPERATION TIMINGS (ms)
  01_questio

In [12]:
session.close()

## Add Articles to Project

In [13]:
from kilin.workspace import add_project_sources

fixtures = repo_root / "tests" / "fixtures"

articles = sorted(fixtures.glob("*.pdf"))

if not articles:
    raise FileNotFoundError(f"No PDF articles found in {fixtures}")

project = add_project_sources(
    project_dir,
    articles,
)

print(f"Project: {project.root}")
print(f"Imported articles: {len(articles)}")

for article in sorted(project.source.glob("*.pdf")):
    print(f"  {article.name}")

Project: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature
Imported articles: 5
  mijalkov17braph.pdf
  mijalkov25computational.pdf
  patil25deep.pdf
  tiunn26braph2.pdf
  yao25connecting.pdf


## Document Discovery and Retrieval

In [14]:
store, conversation, session = create_conversation(project_dir)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 2674.70it/s]


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-10-30T10-07-54.json


In [15]:
answer = ask_question(session, 'Which articles are available in this project?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Which articles are available in this project?

ASSISTANT:
This project has **5 source files**, all with searchable text.

**By type**

- PDF: 5

**5 files**

- mijalkov17braph.pdf (78 parts, 15.8 MB)
- mijalkov25computational.pdf (93 parts, 6.3 MB)
- patil25deep.pdf (114 parts, 6.4 MB)
- tiunn26braph2.pdf (36 parts, 3.1 MB)
- yao25connecting.pdf (130 parts, 3.5 MB)

Counts are of files KíLín selected for this project; a folder of PDFs can hold
tutorials as well as papers.

Wall time: 9.154 s

TURN: turn_0001

EXCHANGE
  question:
Which articles are available in this project?


ANSWER
  summary:
This project has **5 source files**, all with searchable text.


INTENT
  intent:
project_inventory

  standalone:
True

  refers_back:
True

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
none

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
literature

  backend:
mlx

  chat_model:
Qwen

In [16]:
answer = ask_question(session, 'Which of these articles discuss BRAPH?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
Which of these articles discuss BRAPH?

ASSISTANT:
**5 of 5 files** match “discuss BRAPH”, from 741 matching sentences.
- **mijalkov17braph.pdf** (209 sentences): “BRAPH: A graph theory software for
the analysis of brain connectivity” [1]
- **yao25connecting.pdf** (191 sentences): “In this paper, we evaluate the
evidence that genomic results for brain disorders, diseases, and traits point
at speciﬁc brain cell types.” [113]
- **patil25deep.pdf** (136 sentences): “This suggests that while both groups
experienced early stress, bacterial infection” [65]
- **tiunn26braph2.pdf** (124 sentences): “BRAPH 2: A flexible, open-source,
reproducible, community-oriented, easy-to-use framework for network analyses in
neurosciences” [84]
- **mijalkov25computational.pdf** (81 sentences): “By linking the computational
memory capacity of the brain network with cognition, brain function and
integrity, our ﬁndings open new pathways to employ reservoir computing to
investigate aging and age-related 

In [ ]:
answer = ask_question(
    session,
    'Show me the source information for the BRAPH 2 article.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
Show me the source information for the BRAPH 2 article.

ASSISTANT:
**5 of 5 files** match “source information for the BRAPH 2 article”, from 545
matching sentences.
- **yao25connecting.pdf** (218 sentences): “Psychiatry 177, 944–954 (2020).”
[82]
- **mijalkov25computational.pdf** (100 sentences): “Source data for Fig. 2 is
provided as a Source Data ﬁle.” [20]
- **patil25deep.pdf** (93 sentences): “Shade avoidance stress study This
dataset focused on Arabidopsis genotypes responding to” [47]
- **mijalkov17braph.pdf** (85 sentences): “Citation: Mijalkov M, Kakaei E,
Pereira JB, Westman E, Volpe G, for the Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A graph theory software for the analysis of brain
connectivity.” [1]
- **tiunn26braph2.pdf** (49 sentences): “The standard BRAPH 2 distribution
(https://github.com/braph-soft ware/BRAPH-2) provides over 100 fully functional
pipelines for network neuroscience, including single-layer analyses, multilayer
graph analyses inte

In [ ]:
answer = ask_question(
    session,
    'Find passages discussing graph measures in the BRAPH articles.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)



USER:
Find passages discussing graph measures in the BRAPH articles.

ASSISTANT:
**5 of 5 files** match “graph measures”, from 639 matching sentences.
- **mijalkov25computational.pdf** (206 sentences): “This computational memory
capacity is deter- mined by training the network output to reproduce delayed
input time series and comparing the delayed input with the reservoir output,
either across all reservoir nodes (global memory capacity) or across a select
subset (regional memory capacity).” [41]
- **yao25connecting.pdf** (190 sentences): “In this paper, we evaluate the
evidence that genomic results for brain disorders, diseases, and traits point
at speciﬁc brain cell types.” [120]
- **mijalkov17braph.pdf** (148 sentences): “Below we describe in detail the
different options that BRAPH offers for graph theory analy- ses when it comes
to building connectivity matrices, applying threshold strategies, performing
weighted or binary network analyses and computing random networks.” [1]
- **t

In [ ]:
answer = ask_question(
    session,
    'What do those passages say about their implementation?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What do those passages say about their implementation?

ASSISTANT:
<think>
Okay, the user is asking about what the passages say regarding the
implementation of BRAPH. Let me go through the evidence provided.

First, looking at the sources, there are several mentions of BRAPH's
implementation. Source [1] talks about the code being adapted from the Brain
Connectivity Toolbox, which is a key reference. They mention using permutation
tests for nodal measures and adjusting p-values with FDR. Also, they compare
network measures with random graphs, which is part of their implementation.

Source [2] explains the permutation tests in more detail, how they permute
subjects to test group differences and longitudinal data. They also mention
generating random graphs with the same degree or weight distribution, which is
part of their methodology. The user might be interested in how they handle
statistical significance.

Source [4] describes the software architecture with three layers: Graph, 

In [20]:
answer = ask_question(session, 'Which articles discuss deep learning?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Which articles discuss deep learning?

ASSISTANT:
**5 of 5 files** match “deep learning”, from 984 matching sentences.
- **yao25connecting.pdf** (325 sentences): “In this paper, we evaluate the
evidence that genomic results for brain disorders, diseases, and traits point
at speciﬁc brain cell types.” [141]
- **patil25deep.pdf** (265 sentences): “Deep-Learning Investigation of
Vibrational Raman Spec- tra for Plant-Stress Analysis” [78]
- **mijalkov25computational.pdf** (200 sentences): “a n g i s” [35]
- **mijalkov17braph.pdf** (166 sentences): “1 UNAM—National Nanotechnology
Research Center & Department of Physics, Bilkent University, Ankara, Turkey, 2
Department of Neurobiology, Care Sciences and Society, Karolinska Institutet,
Stockholm, Sweden, 3 Department of Physics, Goteborg University, Goteborg,
Sweden” [1]
- **tiunn26braph2.pdf** (28 sentences): “The standard BRAPH 2 distribution
(https://github.com/braph-soft ware/BRAPH-2) provides over 100 fully functional
pipelines fo

In [21]:
session.close()

## Article Understanding and Synthesis

In [22]:
store, conversation, session = create_conversation(project_dir)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 2520.62it/s]


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-10-30T12-39-22.json


In [23]:
answer = ask_question(session, 'Give me an overview of the BRAPH 2 paper.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)



USER:
Give me an overview of the BRAPH 2 paper.

ASSISTANT:
The BRAPH 2 paper introduces a flexible, open-source, and reproducible
framework for network analyses in neuroscience. It addresses the growing
complexity and size of network analyses by providing a versatile platform that
supports advanced statistical models, deep learning, and community-oriented
extension and sharing. The framework, built on MATLAB, allows users to create
specialized distributions with custom pipelines, ensuring flexibility and
scalability across diverse research domains. BRAPH 2 includes a Genesis
compiler that enables users to build and customize pipelines, GUIs, and unit
tests, facilitating integrated statistical testing and reproducible workflows.
The paper demonstrates the framework's capabilities through three case studies:
a standard multiplex graph analysis, a custom deep learning pipeline for
recursive feature elimination, and a standalone distribution for memory
capacity analysis. These examples i

In [ ]:
answer = ask_question(
    session,
    'Summarize its main contributions and limitations.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
Summarize its main contributions and limitations.

ASSISTANT:
<think>
Okay, the user wants a summary of the main contributions and limitations of
BRAPH 2 based on the provided evidence. Let me start by going through each of
the evidence snippets to identify what's mentioned.

First, looking at the contributions. The paper introduces BRAPH 2 as a
flexible, open-source framework for network analysis in neuroscience. It's
built on MATLAB, which allows for custom pipelines and specialized
distributions. The Genesis module is a key feature, enabling users to build and
customize pipelines, GUIs, and unit tests. This modularity and extensibility
are important points. The framework supports both standard graph analysis and
custom software development, as shown in the three case studies.
Reproducibility is a major contribution, with .b2 files that encapsulate all
steps of the pipeline, ensuring exact replication across systems. The
open-source model on GitHub with active community engage

In [25]:
answer = ask_question(
    session,
    'Generate a detailed summary of the BRAPH 2 paper.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
Generate a detailed summary of the BRAPH 2 paper.

ASSISTANT:
The BRAPH 2 paper introduces a flexible, open-source, and reproducible
framework for network analyses in neuroscience. It is designed to support both
standard and custom network analysis pipelines, enabling researchers to perform
complex analyses with ease and transparency. The framework is built on a
modular architecture called Genesis, which allows users to define and compile
custom pipelines, specialized distributions, and graphical user interfaces
(GUIs) with minimal coding. This modular design supports the creation of new
methods and tools, enhancing reproducibility and extensibility across diverse
research domains. BRAPH 2 includes a standard distribution with over 100
pre-built pipelines for network neuroscience, including single-layer and
multilayer graph analyses, and deep learning pipelines for tasks such as
classification and prediction. It also supports the development of standalone
specialized distributio

In [26]:
answer = ask_question(
    session,
    'How does BRAPH 2 differ from the earlier BRAPH framework?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
How does BRAPH 2 differ from the earlier BRAPH framework?

ASSISTANT:
<think>
Okay, the user is asking how BRAPH 2 differs from the earlier BRAPH framework.
Let me look through the provided evidence to find the relevant points.

First, I remember that the evidence mentions Genesis in BRAPH 2. In source [1],
it talks about generating standalone distributions and combining user-defined
elements with standard components. That seems like a key difference because
earlier versions might not have had this modular architecture. Also, the
ability to create custom distributions with minimal coding effort is a new
feature.

Then, looking at [2], the standard distribution in BRAPH 2 has over 100
pipelines, including deep learning ones. The earlier BRAPH might not have had
such a comprehensive set, especially with deep learning integration. Also, the
mention of MATLAB Deep Learning Toolbox in [2] suggests that BRAPH 2 has more
advanced functionalities compared to the previous version.

Sourc

In [27]:
answer = ask_question(session, 'What evidence supports those differences?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What evidence supports those differences?

ASSISTANT:
<think>
Okay, the user is asking for evidence that supports the differences between
BRAPH 2 and the earlier BRAPH framework. Let me go through the provided
evidence to find the relevant parts.

First, looking at the sources, [1] mentions that BRAPH 2 has a modular
architecture called Genesis, allowing standalone specialized distributions with
minimal coding. It also talks about the reservoir computing example, which
wasn't in the earlier version. That's a key point. Then, [2] discusses the
portability across OS and MATLAB versions, along with the open-source model and
GitHub hosting, which might be new features. [3] points out that BRAPH 2 is in
MATLAB, which could be a difference, but also mentions limitations. [4] and [5]
talk about the standard distribution and case studies, showing the new
pipelines and GUI. [6] and [7] explain how Genesis creates specialized
distributions, which is a difference. [8] and [9] mention the F

In [28]:
answer = ask_question(
    session,
    'Which of the four papers is most relevant to graph-based analysis '
    'of brain connectivity, and why?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Which of the four papers is most relevant to graph-based analysis of brain
connectivity, and why?

ASSISTANT:
<think>
Okay, the user is asking which of the four papers is most relevant to
graph-based analysis of brain connectivity and why. Let me start by recalling
the context. The user mentioned four papers, but looking at the evidence
provided, there are actually more than four. Wait, the user might have meant
the four papers listed in the initial question, but in the evidence, there are
multiple sources. Let me check again.

Wait, the user's question is about four papers, but the evidence includes
sources like [1] to [15]. However, the user might have confused the number. Let
me focus on the relevant parts. The user's initial question was about BRAPH 2
and earlier versions, and the evidence includes multiple sources. The user now
is asking which of the four papers is most relevant, but the evidence includes
more than four. Maybe the user is referring to the four papers mentio

In [31]:
session.close()

## Specialized Literature Tools

In [33]:
store, conversation, session = create_conversation(project_dir)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 3467.19it/s]


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-10-30T12-54-32.json


In [34]:
answer = ask_question(
    session,
    'Find the passage in the BRAPH 2 paper '
    'describing its software architecture.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Find the passage in the BRAPH 2 paper describing its software architecture.

ASSISTANT:
The passage in the BRAPH 2 paper describing its software architecture is
mentioned in the notes as being in section 3.1 [18]. Additionally, the BRAPH 2
software architecture is described in a configuration file called
`braph2newdistribution_config.m` [25]. This configuration file, along with
custom `*.gen.m` scripts, defines the behavior and properties of each element
in BRAPH 2. The Genesis module compiles these inputs through a step-by-step
workflow, which includes Directory Setup, File Integration, Element
Compilation, GUI Generation, and Unit Test Creation [25]. The result is a
standalone BRAPH 2 distribution with custom features. The Genesis module is
described as the core innovation of BRAPH 2 [18], enabling the creation of
specialized distributions while preserving core functionality and
reproducibility. The architecture supports unit testing across different
operating systems and MATL

In [35]:
answer = ask_question(
    session,
    'What programming language is BRAPH 2 implemented in?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What programming language is BRAPH 2 implemented in?

ASSISTANT:
The programming language for BRAPH 2 is stated as Python in the document
yao25connecting.pdf [346]. However, another document, tiunn26braph2.pdf,
mentions that BRAPH 2 is implemented in MATLAB [309]. Additionally,
mijalkov25computational.pdf states that BRAPH 2 is implemented in Python [124].
The document patil25deep.pdf does not mention the programming language used for
BRAPH 2 [173]. The document yao25connecting.pdf also mentions BRAPH 2 but does
not specify the programming language it is implemented in [391].

Wall time: 636.302 s

TURN: turn_0002

EXCHANGE
  question:
What programming language is BRAPH 2 implemented in?


ANSWER

INTENT
  intent:
fact_finding

  standalone:
True

  refers_back:
False

  related_turn_ids:
[]


ROUTING
  shortcut:
None

  fallback:
None


RESOLVED POLICY
  history_policy:
relevant

  retrieval_mode:
none

  retrieval_strategy:
hybrid


EFFECTIVE SETTINGS
  project_type_id:
litera

In [36]:
answer = ask_question(session, 'List the articles that mention graph theory.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
List the articles that mention graph theory.

ASSISTANT:
Found **65 mentions** of “graph theory” in **5 of 5 files** (ignoring case,
whole words).

**Mentions per file**

- mijalkov17braph.pdf: 59
- tiunn26braph2.pdf: 3
- mijalkov25computational.pdf: 1
- patil25deep.pdf: 1
- yao25connecting.pdf: 1

**Where**

1. mijalkov17braph.pdf, page 1, line 3: BRAPH: A **graph theory** software for
the analysis of brain connectivity [1]
2. mijalkov17braph.pdf, page 1, lines 27-28: … Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A **graph theory** software for the analysis of brain
connectivity. PLoS ONE 12… [2]
3. mijalkov17braph.pdf, page 1, line 63: …ain network has been studied
extensively using concepts from **graph theory**, where the brain is
represented as a set of nodes connected… [3]
4. mijalkov17braph.pdf, page 1, line 76: …e abilities of BRAPH, we performed
structural and functional **graph theory** analyses in two separate studies. In
the first study, using … [4]
5. 

In [37]:
answer = ask_question(
    session,
    'Filter those results to articles discussing brain connectivity.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Filter those results to articles discussing brain connectivity.

ASSISTANT:
**51 of 65** items from turn 3 match.
1. mijalkov17braph.pdf, line 2: “BRAPH: A graph theory software for the
analysis of brain connectivity” [1]
2. mijalkov17braph.pdf, line 26: “Westman E, Volpe G, for the Alzheimer’s
Disease Neuroimaging Initiative (2017) BRAPH: A graph theory software for the
analysis of brain connectivity. PLoS ONE 12(8): e0178798. https://” [2]
3. mijalkov17braph.pdf, line 62: “its various regions. In the past few years,
the organization of the human brain network has been studied extensively using
concepts from graph theory, where the brain is represented as a set of nodes
connected by edges. This representation of the brain as a connectome” [3]
4. mijalkov17braph.pdf, line 75: “expanded and customized. To demonstrate the
abilities of BRAPH, we performed structural and functional graph theory
analyses in two separate studies. In the first study, using MRI data, we
assessed the dif

In [38]:
answer = ask_question(
    session,
    'Classify the five articles by their main research topic.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
Classify the four articles by their main research topic.

ASSISTANT:
**5 of 5 files** match “main research topic”, from 1353 matching sentences.
- **yao25connecting.pdf** (460 sentences): “In this paper, we evaluate the
evidence that genomic results for brain disorders, diseases, and traits point
at speciﬁc brain cell types.” [197]
- **patil25deep.pdf** (302 sentences): “and diagnostic technologies 1—have
signiﬁcantly improved our ability to monitor plant health.” [120]
- **mijalkov25computational.pdf** (286 sentences): “Here we use reservoir
computing10—a machine-learning framework that can reproduce the functional
dynamics of biological networks11–14—to characterize how the cognitive function
of the human brain deterio- rates with aging.” [58]
- **mijalkov17braph.pdf** (210 sentences): “1 UNAM—National Nanotechnology
Research Center & Department of Physics, Bilkent University, Ankara, Turkey, 2
Department of Neurobiology, Care Sciences and Society, Karolinska Institutet,
Stock

In [39]:
answer = ask_question(
    session,
    'Review the five articles for their relevance to a literature review '
    'on computational brain connectivity.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Review the five articles for their relevance to a literature review on
computational brain connectivity.

ASSISTANT:
**5 of 5 files** match “computational brain connectivity”, from 1310 matching
sentences.
- **yao25connecting.pdf** (362 sentences): “Connecting genomic results for
psychiatric disorders to human brain cell types and regions reveals convergence
with functional connectivity” [189]
- **mijalkov25computational.pdf** (307 sentences): “Computational memory
capacity predicts aging and cognitive decline” [59]
- **mijalkov17braph.pdf** (274 sentences): “BRAPH: A graph theory software for
the analysis of brain connectivity” [1]
- **patil25deep.pdf** (272 sentences): “Raman spectroscopy has been shown to be
able to identify biomolecules at key metabolic centers” [115]
- **tiunn26braph2.pdf** (95 sentences): “The standard BRAPH 2 distribution
(https://github.com/braph-soft ware/BRAPH-2) provides over 100 fully functional
pipelines for network neuroscience, including single-la

In [40]:
answer = ask_question(
    session,
    'Check whether the BRAPH 2 paper supports the claim '
    'that it provides a graphical user interface.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
Check whether the BRAPH 2 paper supports the claim that it provides a graphical
user interface.

ASSISTANT:
<think>
Okay, the user is asking if the BRAPH 2 paper supports the claim that it
provides a graphical user interface. Let me go through the evidence provided.

Looking at the sources, [1] mentions that BRAPH 2 is implemented in MATLAB and
supports multiple data types. It also says the GUI supports users with limited
coding experience. That's a direct mention of GUI.

In [2], there's a part about the GUI supporting users with limited coding
experience and the modular architecture. Also, it mentions the GUI is part of
the software's features.

Source [4] has a figure description where the Genesis module creates the GUI as
part of the distribution. It specifically states that the resulting
distribution includes a no-code GUI for end users. That's a clear support for
the GUI claim.

Source [5] talks about launching the GUI (Fig. 1c) for pipeline selection. The
BRAPH 2 pipeline

In [41]:
answer = ask_question(
    session,
    'Which article should I cite for the introduction of the BRAPH framework?',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which article should I cite for the introduction of the BRAPH framework?

ASSISTANT:
**Statement:** “Which article should I cite for the introduction of the BRAPH
framework?”

**3 of the 5 papers checked could be cited for it:**

1. **Partly supported** · `patil25deep.pdf`
   The work mentions BRAPH2DiSTAP as a cus­tom fork of BRAPH 2.0, but does not
di­rectly ad­dress the need to cite an arti­cle for the in­tro­duc­tion of the
BRAPH frame­work. The cita­tion re­quire­ment is not dis­cussed in the
pas­sages. (Its quote, “data cura­tion for deep learn­ing was car­ried out
us­ing BRAPH2DiSTAP soft­ware, a cus­tom­ized fork of the BRAPH 2.0 soft­ware”,
is not in the passages.)

2. **Partly supported** · `mijalkov25computational.pdf`
   *It says:* “BRAPH 2.0 Memory Capacity. Zenodo https://doi.
org/10.5281/ZENODO” [1]
   The passage mentions BRAPH 2.0 and its memory capacity, but it does not
directly address the introduction of the BRAPH framework. The citation provided
is for a spe

In [42]:
session.close()

## Continuity and Robustness

In [43]:
store, conversation, session = create_conversation(project_dir)

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 3303.72it/s]


Conversation: /Users/giovannivolpe/Documents/GitHub/KiLin/sandbox/trial_literature/logs/traces/conversation_2026-10-30T13-31-05.json


In [44]:
answer = ask_question(session, 'Summarize the BRAPH 2 article.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)


USER:
Summarize the BRAPH 2 article.

ASSISTANT:
The BRAPH 2 article introduces a flexible, open-source, and reproducible
framework for network analyses in neuroscience. It addresses the growing
complexity and size of network analyses by providing a versatile platform that
supports advanced statistical models, deep learning, and community-oriented
extension and sharing. The framework, built on MATLAB, allows users to create
custom pipelines and specialized distributions through its Genesis compiler,
which translates structured element descriptions into executable MATLAB
classes, graphical user interfaces (GUIs), and unit tests. This enables
reproducibility, scalability, and integration across diverse research domains.
The paper demonstrates BRAPH 2's capabilities through three case studies: a
standard multiplex graph analysis of sex differences in brain organization, a
custom deep learning pipeline for recursive feature elimination in
schizophrenia classification, and a standalone dis

In [45]:
answer = ask_question(session, 'What are its main limitations?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)


USER:
What are its main limitations?

ASSISTANT:
<think>
Okay, the user is asking about the main limitations of BRAPH 2 as discussed in
the provided evidence. Let me go through each of the evidence snippets to find
relevant points.

First, looking at evidence [2], it mentions that BRAPH 2 is implemented in
MATLAB, which can limit large voxel-level networks due to memory demands.
Sparse matrices help but reduce computational efficiency. Also, Genesis isn't
suited for direct benchmarking against conventional tools, so benchmarking is
left for future work. That's a key limitation.

Then, evidence [5] talks about the challenges in adopting cutting-edge analyses
for consistency and transparency, which BRAPH 2 aims to solve. But this might
not be a limitation of BRAPH 2 itself but rather a problem the tool addresses.
So maybe not directly relevant.

Evidence [1] mentions the GUI functionality and reproducibility, but doesn't
talk about limitations. The case study example is available, but t

In [46]:
answer = ask_question(session, 'How does it compare with the 2017 paper?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=3,
)


USER:
How does it compare with the 2017 paper?

ASSISTANT:
<think>
Okay, the user is asking how the BRAPH 2 paper compares with the 2017 paper.
Let me start by recalling what I know from the provided evidence.

First, I need to figure out what the 2017 paper might be referring to. The user
mentioned "the 2017 paper," but the evidence here is all from
tiunn26braph2.pdf. The BRAPH 2 paper is from 2026, as indicated by the
SoftwareX 35 (2026) 102791 in source [8]. So the 2017 paper is a different
work, but the evidence here doesn't mention it directly. However, the user
might be referring to BRAPH 1, which is mentioned in the evidence as the
predecessor to BRAPH 2.

Looking at the evidence, there are several references to BRAPH 1. For example,
source [7] mentions that BRAPH 2 adds features like multiplex graph analysis,
deep learning pipelines, and the Genesis module compared to BRAPH 1. Also,
source [2] talks about limitations of BRAPH 2, such as the Genesis module not
being suited for 

In [47]:
answer = ask_question(session, 'Now tell me about the deep learning article.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=4,
)


USER:
Now tell me about the deep learning article.

ASSISTANT:
The paper introduces BRAPH 2, an open-source framework for network analyses in
neuroscience, with a focus on deep learning. It addresses challenges in
reproducibility, maintainability, and long-term reliability in neuroscience
research by providing a flexible, extensible, and reproducible platform. BRAPH
2 includes a Genesis compiler that allows users to build specialized
distributions with custom pipelines, ensuring flexibility and scalability
across diverse research domains. The framework supports both standard and
custom pipelines, enabling users to perform advanced statistical models and
deep learning tasks. The paper demonstrates the application of BRAPH 2 in a
deep learning analysis of recursive feature elimination for schizophrenia
classification, where existing elements from the standard distribution were
reused to create a custom pipeline. This pipeline trained a multilayer
perceptron (MLP) classifier, computed fe

In [48]:
answer = ask_question(session, 'What methods does it use?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=5,
)


USER:
What methods does it use?

ASSISTANT:
<think>
Okay, the user is asking, "What methods does it use?" referring to the BRAPH 2
paper. Let me look through the provided evidence to find the answer.

First, I need to identify which parts of the evidence discuss the methods used
in BRAPH 2. The user mentioned the deep learning article, but the context here
is the BRAPH 2 paper. Let me check the sources.

Looking at source [1], it mentions a custom deep learning pipeline focusing on
recursive feature elimination using a multilayer perceptron and iterative
feature elimination. That's a method. Also, there's mention of the Genesis
module for creating specialized distributions.

Source [2] talks about the standard distribution providing deep learning
pipelines for tasks like classification, implemented in MATLAB with the Deep
Learning Toolbox. So the methods include using MATLAB and the Deep Learning
Toolbox for these pipelines.

Source [5] explains that BRAPH 2 uses the Genesis module to

In [49]:
answer = ask_question(session, 'Compare it with the previous article.')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=6,
)


USER:
Compare it with the previous article.

ASSISTANT:
<think>
Okay, the user is asking to compare the current paper (which is about BRAPH 2)
with the previous article, which I assume refers to the 2017 paper mentioned
earlier. Let me go through the provided evidence to find the relevant points.

First, the user wants a point-by-point comparison. The previous article was the
2017 paper, which I know from the initial conversation was BRAPH 1. The current
paper is BRAPH 2. Let me check the evidence for differences and similarities.

Looking at the sources, [1] and [2] mention that BRAPH 2 introduces multiplex
graph analysis, deep learning pipelines, and the Genesis module for generating
custom workflows. The 2017 paper focused on foundational network analysis and
ROI-based connectivity. So that's a key difference in functionality. Also,
BRAPH 2 emphasizes reproducibility through .b2 files, which the 2017 paper
might not have had. [4] and [9] talk about the Genesis module and how it all

In [50]:
answer = ask_question(
    session,
    'Find evidence that all five articles use the same dataset.',
)

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=7,
)


USER:
Find evidence that all five articles use the same dataset.

ASSISTANT:
**5 of 5 files** match “use the same dataset”, from 733 matching sentences.
- **yao25connecting.pdf** (211 sentences): “Methods Human Brain Atlas
single-nucleus RNA-seq (snRNAseq) We used the Human Brain Atlas snRNAseq data
set from Siletti et al.27.” [114]
- **patil25deep.pdf** (199 sentences): “The latent space derived from the
24-hour post-inﬁltration data is shown in Fig. 5a.” [72]
- **mijalkov25computational.pdf** (196 sentences): “Computational memory
capacity predicts aging and cognitive decline” [26]
- **mijalkov17braph.pdf** (112 sentences): “Citation: Mijalkov M, Kakaei E,
Pereira JB, Westman E, Volpe G, for the Alzheimer’s Disease Neuroimaging
Initiative (2017) BRAPH: A graph theory software for the analysis of brain
connectivity.” [1]
- **tiunn26braph2.pdf** (15 sentences): “All results and analysis steps can be
reproduced using the provided .b2 files, available at
https://github.com/braph-softwar

In [51]:
answer = ask_question(session, 'Which article did we discuss first?')

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=8,
)


USER:
Which article did we discuss first?

ASSISTANT:
<think>
Okay, the user is asking which article was discussed first. Let me check the
provided evidence.

Looking at the evidence, there are multiple files mentioned, but the user is
referring to the BRAPH 2 article. The initial question was about summarizing
the BRAPH 2 article, and then comparing it with the previous one. The user then
asked if all five articles use the same dataset, and the assistant confirmed
that. Now, the user is asking which article was discussed first.

In the first interaction, the user mentioned the BRAPH 2 article, which is
tiunn26braph2.pdf. The assistant provided a summary of that article. Then, when
comparing with the previous article, the assistant referred to BRAPH 1, which
is likely the 2017 paper by Mijalkov et al. (mijalkov17braph.pdf). However, the
user's current question is about which article was discussed first, and the
evidence provided includes the BRAPH 2 article as the first one discussed.

In [52]:
session.close()